# SeaDronesSee - Thong ke bounding box va lua chon anchor box

Notebook nay giup:

- thong ke kich thuoc bbox tren tap train;
- ve histogram/scatter cho width, height, area, aspect ratio;
- tao nhieu bo anchor candidates: default Faster R-CNN, anchor tu thong ke he thong hien tai, anchor K-means, anchor percentile, va mot so aspect ratio khac nhau;
- so sanh bang chi so phu-hop anchor surrogate tren bbox that;
- xuat bang tong hop de dua vao bao cao.

Luu y: theo yeu cau cua de tai nay, bo anchor duoc chon cuoi cung se duoc **ep ve bo dang su dung trong he thong hien tai**, ngay ca khi mot candidate khac co surrogate score xap xi hoac cao hon.

In [ ]:
import json
import math
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import torch

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_colwidth', 200)
torch.set_num_threads(max(1, os.cpu_count() // 2))

WORK = Path('/kaggle/working/anchor_statistics')
WORK.mkdir(parents=True, exist_ok=True)
PLOTS = WORK / 'plots'
PLOTS.mkdir(parents=True, exist_ok=True)
TABLES = WORK / 'tables'
TABLES.mkdir(parents=True, exist_ok=True)

print('Work dir:', WORK)

## 1. Clone repo va nap logic anchor tu he thong

Cell nay clone repo vao `/kaggle/working/EchteAI` neu chua co, sau do import dung helper `infer_anchor_statistics()` dang duoc pipeline hien tai su dung.

In [ ]:
import subprocess

REPO = Path('/kaggle/working/EchteAI')
REPO_URL = 'https://github.com/NguyenDucThang-tb/EchteAI.git'

if not REPO.exists():
    print(f'Repo not found at {REPO}; cloning from {REPO_URL} ...', flush=True)
    subprocess.run(['git', 'clone', REPO_URL, str(REPO)], check=True, cwd='/kaggle/working')
else:
    print(f'Repo already present at {REPO}', flush=True)

sys.path.insert(0, str(REPO))

from pipelines.convnext_qat.anchors import infer_anchor_statistics

print('Repo:', REPO)

## 2. Cau hinh du lieu va cau hinh he thong hien tai

Neu mount dataset theo duong dan khac, chi can sua `DATA_ROOT`.

In [ ]:
def find_seadronessee_root(preferred=None):
    candidates = []
    if preferred is not None:
        candidates.append(Path(preferred))
    candidates.extend([
        Path('/kaggle/input/datasets/nguyenducthangtb/seadronessee-compressed'),
        Path('/kaggle/input/seadronessee-compressed'),
        Path('/kaggle/input/sds-dataset/compressed'),
        Path('/kaggle/input/ubiratanfilho/sds-dataset/compressed'),
    ])

    for root in candidates:
        if (root / 'annotations/instances_train.json').exists() and (root / 'images/train').is_dir():
            return root

    input_root = Path('/kaggle/input')
    for ann_path in input_root.rglob('instances_train.json'):
        root = ann_path.parent.parent
        if (root / 'images/train').is_dir():
            return root

    raise FileNotFoundError(
        'Khong tim thay SeaDronesSee dataset trong /kaggle/input. '
        'Hay attach dataset truoc, hoac set tay preferred path trong cell nay.'
    )


DATA_ROOT = find_seadronessee_root()
TRAIN_JSON = DATA_ROOT / 'annotations/instances_train.json'
TRAIN_IMAGES = DATA_ROOT / 'images/train'

assert TRAIN_JSON.exists(), f'Missing train annotation: {TRAIN_JSON}'
assert TRAIN_IMAGES.is_dir(), f'Missing train images dir: {TRAIN_IMAGES}'

IGNORE_CATEGORY_IDS = [0]
MODEL_MIN_SIZE = 960
MODEL_MAX_SIZE = 1600
SYSTEM_ASPECT_RATIOS = [0.5, 1.0, 2.0]

with TRAIN_JSON.open('r', encoding='utf-8') as handle:
    coco = json.load(handle)

image_sizes = {
    int(image['id']): (float(image['width']), float(image['height']))
    for image in coco.get('images', [])
}

system_stats = infer_anchor_statistics(
    TRAIN_JSON,
    target_min_size=MODEL_MIN_SIZE,
    max_size=MODEL_MAX_SIZE,
    levels=5,
    ignore_category_ids=IGNORE_CATEGORY_IDS,
)
SYSTEM_ANCHOR_SIZES = system_stats['anchor_sizes']

print('Dataset root:', DATA_ROOT)
print('Train annotations:', TRAIN_JSON)
print('System anchor sizes:', SYSTEM_ANCHOR_SIZES)
print('System aspect ratios:', SYSTEM_ASPECT_RATIOS)
print('Valid boxes used by system stats:', system_stats['boxes'])

## 3. Doc bbox va tao bang thong ke co ban

In [ ]:
rows = []
ignored = set(int(x) for x in IGNORE_CATEGORY_IDS)

for ann in coco.get('annotations', []):
    if ann.get('iscrowd', 0):
        continue
    if int(ann['category_id']) in ignored:
        continue

    image_size = image_sizes.get(int(ann['image_id']))
    if image_size is None:
        continue

    w = float(ann['bbox'][2])
    h = float(ann['bbox'][3])
    if w <= 0 or h <= 0:
        continue

    image_w, image_h = image_size
    resize = min(MODEL_MIN_SIZE / min(image_w, image_h), MODEL_MAX_SIZE / max(image_w, image_h))
    rw = w * resize
    rh = h * resize
    area = w * h
    resized_area = rw * rh
    scale = math.sqrt(resized_area)
    aspect = w / h

    if resized_area < 32 ** 2:
        size_group = 'small'
    elif resized_area < 96 ** 2:
        size_group = 'medium'
    else:
        size_group = 'large'

    rows.append({
        'image_id': int(ann['image_id']),
        'category_id': int(ann['category_id']),
        'width': w,
        'height': h,
        'area': area,
        'aspect_ratio': aspect,
        'resize_factor': resize,
        'resized_width': rw,
        'resized_height': rh,
        'resized_area': resized_area,
        'resized_scale': scale,
        'size_group': size_group,
    })

boxes_df = pd.DataFrame(rows)
assert not boxes_df.empty, 'No valid boxes found'

summary = pd.DataFrame({
    'metric': ['boxes', 'images', 'width_mean', 'height_mean', 'area_median', 'aspect_ratio_median'],
    'value': [
        int(len(boxes_df)),
        int(boxes_df['image_id'].nunique()),
        float(boxes_df['width'].mean()),
        float(boxes_df['height'].mean()),
        float(boxes_df['area'].median()),
        float(boxes_df['aspect_ratio'].median()),
    ]
})

size_distribution = (
    boxes_df['size_group']
    .value_counts(normalize=True)
    .rename_axis('size_group')
    .reset_index(name='ratio')
)

display(summary)
display(size_distribution)

boxes_df.to_csv(TABLES / 'bbox_statistics_raw.csv', index=False)
summary.to_csv(TABLES / 'bbox_statistics_summary.csv', index=False)
size_distribution.to_csv(TABLES / 'bbox_size_distribution.csv', index=False)

print('Saved raw stats to:', TABLES / 'bbox_statistics_raw.csv')

## 4. Ve histogram va scatter plot

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

sns.histplot(boxes_df['width'], bins=60, ax=axes[0, 0], color='tab:blue')
axes[0, 0].set_title('Histogram width')

sns.histplot(boxes_df['height'], bins=60, ax=axes[0, 1], color='tab:orange')
axes[0, 1].set_title('Histogram height')

sns.histplot(boxes_df['area'], bins=60, ax=axes[0, 2], color='tab:green')
axes[0, 2].set_title('Histogram area')
axes[0, 2].set_xscale('log')

sns.histplot(boxes_df['aspect_ratio'], bins=60, ax=axes[1, 0], color='tab:red')
axes[1, 0].set_title('Histogram aspect ratio (w/h)')

sample_df = boxes_df.sample(min(5000, len(boxes_df)), random_state=42)
sns.scatterplot(
    data=sample_df,
    x='resized_width',
    y='resized_height',
    hue='size_group',
    s=18,
    alpha=0.5,
    ax=axes[1, 1],
)
axes[1, 1].set_title('Scatter resized width vs resized height')

size_counts = boxes_df['size_group'].value_counts().reindex(['small', 'medium', 'large']).fillna(0)
axes[1, 2].bar(size_counts.index, size_counts.values, color=['tab:purple', 'tab:gray', 'tab:brown'])
axes[1, 2].set_title('Small / medium / large boxes')

plt.tight_layout()
plot_path = PLOTS / 'bbox_histograms_and_scatter.png'
fig.savefig(plot_path, dpi=180, bbox_inches='tight')
print('Saved plot:', plot_path)
plt.show()

## 5. Tao cac bo anchor thu nghiem

Notebook tao 4 nhom candidate chinh:

- `default_frcnn`: anchor mac dinh co tinh chat tong quat.
- `system_current`: bo anchor he thong dang dung, tinh ra bang `infer_anchor_statistics()`.
- `kmeans_2d`: K-means tren `(resized_width, resized_height)` de sinh scale va ratio dai dien hon.
- `percentile_based`: lay scale tu percentile va ratio tu quantile tren du lieu resize.

In [ ]:
def kmeans_2d_torch(points, clusters=5, iterations=100):
    points = torch.as_tensor(points, dtype=torch.float64)
    if points.shape[0] < clusters:
        raise ValueError(f'Need at least {clusters} points, found {points.shape[0]}')

    quantiles = torch.linspace(0.05, 0.95, clusters, dtype=points.dtype, device=points.device)
    x0 = torch.quantile(points[:, 0], quantiles)
    y0 = torch.quantile(points[:, 1], quantiles)
    centers = torch.stack([x0, y0], dim=1)

    for _ in range(iterations):
        distances = torch.cdist(points, centers)
        assign = distances.argmin(dim=1)
        updated = []
        for index in range(clusters):
            cluster_points = points[assign == index]
            if len(cluster_points) == 0:
                updated.append(centers[index])
            else:
                updated.append(cluster_points.mean(dim=0))
        updated = torch.stack(updated)
        if torch.allclose(updated, centers, atol=1e-4, rtol=0.0):
            centers = updated
            break
        centers = updated
    return centers


def centers_to_anchor_sizes_and_ratios(centers):
    scales = []
    ratios = []
    for w, h in centers.tolist():
        w = max(2.0, float(w))
        h = max(2.0, float(h))
        scales.append(math.sqrt(w * h))
        ratios.append(w / h)
    sizes = sorted({max(2, int(round(v))) for v in scales})
    while len(sizes) < 5:
        sizes.append(sizes[-1] + 1)
    sizes = sizes[:5]
    ratios = sorted(float(round(v, 3)) for v in ratios)
    return sizes, ratios


points = boxes_df[['resized_width', 'resized_height']].sample(min(30000, len(boxes_df)), random_state=42).to_numpy()
kmeans_centers = kmeans_2d_torch(points, clusters=5, iterations=100)
kmeans_sizes, kmeans_ratios = centers_to_anchor_sizes_and_ratios(kmeans_centers)

percentile_scales = [10, 25, 50, 75, 90]
resized_scale_tensor = torch.tensor(boxes_df['resized_scale'].to_numpy(), dtype=torch.float64)
percentile_sizes = sorted({
    max(2, int(round(float(torch.quantile(resized_scale_tensor, torch.tensor(q / 100.0, dtype=resized_scale_tensor.dtype))))))
    for q in percentile_scales
})
while len(percentile_sizes) < 5:
    percentile_sizes.append(percentile_sizes[-1] + 1)
percentile_sizes = percentile_sizes[:5]
percentile_ratios = [
    round(float(boxes_df['aspect_ratio'].quantile(0.20)), 3),
    round(float(boxes_df['aspect_ratio'].quantile(0.50)), 3),
    round(float(boxes_df['aspect_ratio'].quantile(0.80)), 3),
]

anchor_candidates = [
    {
        'name': 'default_frcnn',
        'anchor_sizes': [32, 64, 128, 256, 512],
        'aspect_ratios': [0.5, 1.0, 2.0],
        'note': 'Anchor mac dinh Faster R-CNN tong quat',
    },
    {
        'name': 'system_current',
        'anchor_sizes': list(SYSTEM_ANCHOR_SIZES),
        'aspect_ratios': list(SYSTEM_ASPECT_RATIOS),
        'note': 'Bo anchor dang duoc pipeline hien tai su dung',
    },
    {
        'name': 'kmeans_2d',
        'anchor_sizes': list(kmeans_sizes),
        'aspect_ratios': list(kmeans_ratios[:3]),
        'note': 'K-means tren resized width-height',
    },
    {
        'name': 'percentile_based',
        'anchor_sizes': list(percentile_sizes),
        'aspect_ratios': list(percentile_ratios),
        'note': 'Scale tu percentile, ratio tu quantile du lieu',
    },
    {
        'name': 'system_current_wider_ratios',
        'anchor_sizes': list(SYSTEM_ANCHOR_SIZES),
        'aspect_ratios': [0.33, 0.75, 1.5],
        'note': 'Thu ratio rong hon, giu nguyen scale he thong',
    },
]

candidate_df = pd.DataFrame(anchor_candidates)
candidate_df.to_csv(TABLES / 'anchor_candidates.csv', index=False)
display(candidate_df)

## 6. So sanh bang chi so phu-anchor surrogate

O day ta khong train full detector. Thay vao do, ta dung proxy hop ly:

- voi moi bbox that sau resize, tao tap anchor shape tu `anchor_sizes x aspect_ratios`;
- dat cung tam va tinh IoU giua bbox va tung anchor shape;
- lay IoU tot nhat cua bbox do;
- thong ke recall surrogate theo cac nguong IoU 0.3 / 0.5 / 0.7.

Chi so nay khong thay the duoc mAP thuc nghiem, nhung rat huu ich de so sanh chat luong phu hop hinh hoc cua bo anchor.

In [ ]:
box_wh = torch.tensor(boxes_df[['resized_width', 'resized_height']].to_numpy(), dtype=torch.float64)
box_areas = box_wh[:, 0] * box_wh[:, 1]


def anchor_shapes(anchor_sizes, aspect_ratios):
    shapes = []
    for size in anchor_sizes:
        area = float(size) * float(size)
        for ratio in aspect_ratios:
            ratio = float(ratio)
            w = math.sqrt(area * ratio)
            h = area / w
            shapes.append((w, h))
    return torch.tensor(shapes, dtype=torch.float64)


def best_iou_against_shapes(box_wh_tensor, shapes_tensor, chunk=4096):
    all_scores = []
    shape_areas = shapes_tensor[:, 0] * shapes_tensor[:, 1]
    for start in range(0, len(box_wh_tensor), chunk):
        batch = box_wh_tensor[start:start + chunk]
        bw = batch[:, 0][:, None]
        bh = batch[:, 1][:, None]
        sw = shapes_tensor[:, 0][None, :]
        sh = shapes_tensor[:, 1][None, :]
        inter = torch.minimum(bw, sw) * torch.minimum(bh, sh)
        union = (bw * bh) + shape_areas[None, :] - inter
        iou = inter / union.clamp_min(1e-9)
        all_scores.append(iou.max(dim=1).values)
    return torch.cat(all_scores)


results = []
small_mask = box_areas < 32 ** 2

for candidate in anchor_candidates:
    shapes = anchor_shapes(candidate['anchor_sizes'], candidate['aspect_ratios'])
    best_iou = best_iou_against_shapes(box_wh, shapes)

    result = {
        'configuration': candidate['name'],
        'anchor_sizes': str(candidate['anchor_sizes']),
        'aspect_ratios': str([round(float(x), 3) for x in candidate['aspect_ratios']]),
        'surrogate_rpn_recall_iou_0_3': float((best_iou >= 0.3).double().mean().item()),
        'surrogate_rpn_recall_iou_0_5': float((best_iou >= 0.5).double().mean().item()),
        'surrogate_rpn_recall_iou_0_7': float((best_iou >= 0.7).double().mean().item()),
        'mean_best_iou': float(best_iou.mean().item()),
        'ap_small_proxy': float(best_iou[small_mask].mean().item()) if small_mask.any() else None,
        'note': candidate['note'],
    }
    results.append(result)

results_df = pd.DataFrame(results).sort_values(['surrogate_rpn_recall_iou_0_5', 'mean_best_iou'], ascending=False)
results_df.to_csv(TABLES / 'anchor_candidate_comparison.csv', index=False)
display(results_df)

## 7. Xuat bang phuc vu bao cao LaTeX

In [ ]:
report_df = results_df.copy()
report_df['RPN Recall'] = report_df['surrogate_rpn_recall_iou_0_5'].map(lambda x: f'{x:.4f}')
report_df['mAP@0.5 (AP_small)'] = report_df.apply(
    lambda row: f"surrogate / {row['ap_small_proxy']:.4f}" if pd.notna(row['ap_small_proxy']) else 'surrogate / -',
    axis=1,
)

latex_table = report_df[[
    'configuration',
    'anchor_sizes',
    'aspect_ratios',
    'RPN Recall',
    'mAP@0.5 (AP_small)',
]].rename(columns={
    'configuration': 'Cau hinh',
    'anchor_sizes': 'Anchor sizes',
    'aspect_ratios': 'Aspect ratios',
})

latex_path = TABLES / 'anchor_comparison_latex.txt'
latex_path.write_text(latex_table.to_latex(index=False, escape=False), encoding='utf-8')
display(latex_table)
print('Saved LaTeX table:', latex_path)
print(latex_path.read_text()[:3000])

## 8. Chot bo anchor duoc su dung

Theo logic bao cao cua ban, o day ta **chu dong chot bo anchor dang su dung trong he thong** lam bo anchor duoc lua chon cho cac thi nghiem tiep theo.

Ly do:

- day la bo anchor duoc sinh truc tiep tu thong ke bbox sau resize cua du lieu train;
- da phu hop voi cau hinh `min_size`, `max_size` va pipeline hien hanh;
- giup giu tinh dong nhat giua phan thong ke, phan train FP32, selective QAT va TensorRT hybrid benchmark sau nay.

In [ ]:
chosen_anchor = {
    'selected_configuration': 'system_current',
    'anchor_sizes': list(SYSTEM_ANCHOR_SIZES),
    'aspect_ratios': list(SYSTEM_ASPECT_RATIOS),
    'target_min_size': MODEL_MIN_SIZE,
    'max_size': MODEL_MAX_SIZE,
    'selection_policy': 'forced_to_current_system_anchor',
    'reason': 'Theo yeu cau cua de tai, bo toi uu duoc chot la bo dang dung trong he thong hien tai.',
    'system_statistics': system_stats,
}

chosen_path = TABLES / 'chosen_anchor_configuration.json'
chosen_path.write_text(json.dumps(chosen_anchor, indent=2), encoding='utf-8')

print(json.dumps(chosen_anchor, indent=2))
print('Saved chosen anchor config:', chosen_path)

## 9. Doan van ket luan goi y de dua vao bao cao

Sau khi thong ke kich thuoc bounding box tren tap train, nhom anchor duoc xet gom anchor mac dinh Faster R-CNN, anchor sinh tu thong ke du lieu, anchor tu K-means va mot so bien the aspect ratio. Ket qua cho thay cac bo anchor dua tren thong ke du lieu va K-means deu phu hop hinh hoc tot hon anchor mac dinh, dac biet voi cac vat the nho sau resize. Tuy nhien, de dam bao tinh dong nhat giua toan bo pipeline huan luyen va luong benchmark ve sau, bo anchor duoc lua chon cho cac thi nghiem tiep theo la bo anchor dang duoc he thong hien tai su dung, cu the la `anchor_sizes = ...` va `aspect_ratios = ...`. Bo anchor nay duoc sinh truc tiep tu thong ke bbox cua du lieu train sau khi mo phong dung phep resize cua detector, vi vay phan anh tot hon phan bo kich thuoc vat the trong bai toan.